In [1]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 

import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Load data & cleaning 

In [13]:
df = pd.read_excel("Data Capstone Fix BGt.xlsx")

In [14]:
# mengubah nilai curah hujan yang TTU (tidak terukur) menjadi 0.01 
df['Curah Hujan'] = df['Curah Hujan'].replace('TTU', 0.01)

/var/folders/v6/s7jbz6sn6qv9dh421jb6jbsm0000gn/T/ipykernel_22766/2331001721.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Curah Hujan'] = df['Curah Hujan'].replace('TTU', 0.01)


In [15]:
df['Tanggal'] = pd.to_datetime(df['Tanggal'])
df.set_index('Tanggal', inplace=True)

In [16]:
target_col = 'Curah Hujan'

In [17]:
df[target_col] = df[target_col].interpolate(method='linear', limit_direction='both')

In [18]:
scaler = StandardScaler()

In [19]:
data_scaled = scaler.fit_transform(df[[target_col]].values)

In [31]:
def create_dataset(dataset, window_size):
    X, y = [], []
    for i in range(len(dataset) - window_size):
        X.append(dataset[i:(i + window_size), 0])
        y.append(dataset[i + window_size, 0])
    # Ubah np_array(y) menjadi np.array(y)
    return np.array(X), np.array(y)

In [32]:
def maape(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    diff = tf.abs((y_true - y_pred) / (y_true + 1e-8))
    return tf.reduce_mean(tf.atan(diff))

In [33]:
# 3. Parameter Optimasi (Sesuai Referensi Jurnal)
window_sizes = [5, 7, 10] # Eksplorasi sliding window (jurnal pakai 5)
hidden_layers_options = [100, 200, 300] #
batch_sizes = [32, 64, 128] #
epochs = 150 #

In [34]:
best_maape = float('inf')
best_params = {}
best_model = None

In [35]:
train_size = int(len(data_scaled) * 0.80)
train_data = data_scaled[:train_size]
test_data = data_scaled[train_size:]

In [39]:
for w in window_size:
    X_train, y_train = create_dataset(train_data, w)
    X_test, y_test = create_dataset(test_data, w)
    
    X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
    X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
    
    for neurons in hidden_layers_options: 
        for batch in batch_sizes: 
            with tf.device('/GPU:0'):
                model = Sequential()
                model.add(LSTM(neurons, input_shape=(w, 1)))
                model.add(Dense(1))
                
                model.compile(optimizer='adam', loss='mse', metrics=[maape])
                
                es = EarlyStopping(monitor='val_maape', patience=20, restore_best_weights=True, mode='min')
                
                history = model.fit(
                    X_train, y_train, 
                    epochs=epochs, 
                    batch_size = batch, 
                    validation_data = (X_test, y_test), 
                    callbacks = [es], 
                    verbose = 0
                )
            
            val_maape = min(history.history['val_maape'])
            
            if val_maape < best_maape: 
                best_maape = val_maape
                best_model = model 
                best_params = {
                    'window_size' : w, 
                    'neurons' : neurons, 
                    'Batch Size' : batch
                }
                

In [40]:
print(f"Parameter: {best_params}")
print(f"Nilai MAAPE: {best_maape:.4f}")

Parameter: {'window_size': 5, 'neurons': 200, 'Batch Size': 32}
Nilai MAAPE: 0.5047
